# Meth3D-Net **V6** — Full-Genome Pipeline

## What went wrong in V5 and how V6 fixes it

### Root cause of Hi-C V5 coverage = 0.0%
The V5 proxy computed Pearson correlation between bin vectors.
When a bin contained NaN-imputed values (bins with no CpG coverage,
e.g. centromeric regions), `my_vec - my_vec.mean()` produced a
zero or NaN vector, making every correlation = 0.
The filter `score > 0` then discarded all contacts → empty contact list
for every bin → `hic_feat` = all zeros → 0% coverage.

### V6 Hi-C proxy — three concrete changes
| Issue | V5 | V6 Fix |
|---|---|---|
| NaN bins | NaN propagates to corr = NaN/0 | Fill NaN bins with **global chromosome mean before correlation** |
| `score > 0` filter | Discards anticorrelated bins (biologically valid) | Keep **top-K by absolute correlation** \|corr\|, sign encoded separately |
| Flat vector = 0 corr | Zero-variance bins contribute nothing | Skip zero-variance bins; assign **distance-decay fallback contacts** |

### V6 also fixes Panels C and D (DMB / CT empty)
The CT score used `ct_sig_99 = ct_score > percentile(ct_score, 99)` but when
`ct_score` was computed on a beta matrix with very small variance
(model-imputed values compressed toward the mean), all 99th-percentile
thresholds were near-zero → all windows flagged or none flagged.
V6 normalises CT score per chromosome using a **z-score** against a
genome-wide baseline, and the RaM block size is auto-tuned to chromosome size.

### Three new publication figures
| Figure | Description |
|---|---|
| **Figure 1** | Genome-wide prediction accuracy — Pearson r, R², MAE per chromosome + V5 vs V6 delta |
| **Figure 2** | Genome-wide instability landscape — DMB density + CT score track + CERS heatmap across all chromosomes |
| **Figure 3** | HLA locus zoom (chr6 25–35 Mb) — methylation tracks, DMBs, CT score, co-methylation network edges |

### All outputs zipped
Every per-chromosome CSV, JSON, PT, PDF and PNG is collected into
`/kaggle/working/V6_results/V6_outputs.zip` at the end.

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 0 — Environment
# ═══════════════════════════════════════════════════════════════════
import os, sys, zipfile
import torch

WORK_DIR    = '/kaggle/working'
RESULTS_DIR = f'{WORK_DIR}/V6_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  No GPU — enable T4 in Settings → Accelerator')
print(f'Output  : {RESULTS_DIR}')

Python  : 3.12.12
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB
Output  : /kaggle/working/V6_results


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ═══════════════════════════════════════════════════════════════════
import gzip, gc, json
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import uniform_filter1d
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Configuration
# ═══════════════════════════════════════════════════════════════════
BASE_GEO = 'https://ftp.ncbi.nlm.nih.gov/geo/samples'

SAMPLES = {
    'GSM429321': {'local': f'{WORK_DIR}/GSM429321_H1_1.wig.gz',
                  'url'  : f'{BASE_GEO}/GSM429nnn/GSM429321/suppl/GSM429321_UCSD.H1.Bisulfite-Seq.combined.wig.gz'},
    'GSM429322': {'local': f'{WORK_DIR}/GSM429322_H1_2.wig.gz',
                  'url'  : f'{BASE_GEO}/GSM429nnn/GSM429322/suppl/GSM429322_UCSD.H1.Bisulfite-Seq.combined.wig.gz'},
    'GSM432687': {'local': f'{WORK_DIR}/GSM432687_IMR90_1.wig.gz',
                  'url'  : f'{BASE_GEO}/GSM432nnn/GSM432687/suppl/GSM432687_UCSD.IMR90.Bisulfite-Seq.combined.wig.gz'},
    'GSM432688': {'local': f'{WORK_DIR}/GSM432688_IMR90_2.wig.gz',
                  'url'  : f'{BASE_GEO}/GSM432nnn/GSM432688/suppl/GSM432688_UCSD.IMR90.Bisulfite-Seq.combined.wig.gz'},
}
PRIMARY_GSMS = ['GSM429321','GSM429322','GSM432687','GSM432688']

CENTROMERES = {
    'chr1':125_000_000,'chr2':93_300_000,'chr3':91_000_000,
    'chr4':50_400_000,'chr5':48_400_000,'chr6':61_000_000,
    'chr7':59_900_000,'chr8':45_600_000,'chr9':49_000_000,
    'chr10':40_200_000,'chr11':53_700_000,'chr12':35_800_000,
    'chr13':17_900_000,'chr14':17_600_000,'chr15':19_000_000,
    'chr16':36_600_000,'chr17':24_000_000,'chr18':17_200_000,
    'chr19':26_500_000,'chr20':27_500_000,'chr21':13_200_000,
    'chr22':14_700_000,'chrX':60_600_000,'chrY':12_300_000,
}

HOTSPOTS = {
    'chr1':{'RUNX3':(25_160_000,25_232_000),'TP73':(3_569_000,3_652_000),'MDM4':(204_496_000,204_630_000)},
    'chr2':{'DNMT3A':(25_455_000,25_565_000),'IDH1':(209_113_000,209_127_000),'MSH2':(47_630_000,47_790_000)},
    'chr3':{'VHL':(10_183_000,10_195_000),'PIK3CA':(178_866_000,178_957_000),'MLH1':(37_034_000,37_093_000)},
    'chr4':{'FGFR3':(1_795_000,1_810_000),'KIT':(55_524_000,55_606_000),'TET2':(106_067_000,106_200_000)},
    'chr5':{'APC':(112_043_000,112_181_000),'TERT':(1_253_000,1_295_000),'NPM1':(170_814_000,170_838_000)},
    'chr6':{'PRDM1':(106_522_000,106_600_000),'HLA':(28_477_000,33_448_000),'HACE1':(102_243_000,102_434_000)},
    'chr7':{'EGFR':(55_086_000,55_324_000),'MET':(116_312_000,116_438_000),'BRAF':(140_453_000,140_624_000)},
    'chr8':{'MYC':(128_748_000,128_753_000),'FGFR1':(38_411_000,38_468_000)},
    'chr9':{'CDKN2A':(21_968_000,21_995_000),'CDKN2B':(22_002_000,22_009_000),'ABL1':(133_590_000,133_763_000)},
    'chr10':{'PTEN':(89_623_000,89_731_000),'RET':(43_572_000,43_625_000)},
    'chr11':{'WT1':(32_409_000,32_457_000),'ATM':(108_093_000,108_340_000),'CCND1':(69_165_000,69_178_000)},
    'chr12':{'KRAS':(25_358_000,25_403_000),'CDK4':(58_142_000,58_146_000),'MDM2':(69_202_000,69_239_000)},
    'chr13':{'BRCA2':(32_889_000,32_974_000),'RB1':(48_877_000,49_057_000)},
    'chr14':{'AKT1':(104_769_000,104_795_000),'DICER1':(95_068_000,95_163_000)},
    'chr15':{'IDH2':(90_631_000,90_644_000),'MAP2K1':(66_679_000,66_730_000)},
    'chr16':{'CDH1':(68_771_000,68_870_000),'PALB2':(23_614_000,23_652_000)},
    'chr17':{'TP53':(7_565_000,7_590_000),'BRCA1':(43_044_000,43_126_000),'NF1':(31_094_000,31_377_000),'ERBB2':(37_844_000,37_886_000)},
    'chr18':{'SMAD4':(48_556_000,48_611_000),'DCC':(18_218_000,18_620_000)},
    'chr19':{'STK11':(1_205_000,1_228_000),'ERCC2':(45_867_000,45_909_000)},
    'chr20':{'ASXL1':(31_022_000,31_066_000),'GNAS':(57_414_000,57_486_000)},
    'chr21':{'RUNX1':(34_787_000,36_004_000),'ERG':(38_380_000,38_677_000)},
    'chr22':{'NF2':(29_999_000,30_094_000),'BCR':(23_179_000,23_318_000)},
    'chrX':{'MECP2':(153_296_000,153_363_000),'AR':(66_764_000,66_951_000),'XIST':(73_040_000,73_072_000)},
    'chrY':{'SRY':(2_655_000,2_699_000),'DAZ1':(25_403_000,25_432_000)},
}

WINDOW_SIZE = 21; CENTER = 10; STRIDE = 3
BATCH_SIZE  = 512; EPOCHS = 20
HIC_K = 5; HIC_DIM = 40; HIC_OUT = 64; POS_DIM = 8
RESOLUTION = 25_000

CHROMS_ALL = [
    'chr1','chr2','chr3','chr4','chr5','chr6','chr7','chr8',
    'chr9','chr10','chr11','chr12','chr13','chr14','chr15',
    'chr16','chr17','chr18','chr19','chr20','chr21','chr22','chrX','chrY'
]
print(f'Config loaded. Chromosomes: {len(CHROMS_ALL)}')

Config loaded. Chromosomes: 24


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Download WIG files
# ═══════════════════════════════════════════════════════════════════
print('Checking WIG files...')
for gsm, info in SAMPLES.items():
    local = info['local']
    if os.path.exists(local):
        mb = os.path.getsize(local)/1e6
        if mb < 1.0:
            print(f'  {gsm} {mb:.1f}MB incomplete — re-downloading')
            os.remove(local)
        else:
            print(f'  {gsm} {mb:.0f}MB ✓'); continue
    print(f'  {gsm} downloading (~220MB)...')
    ret = os.system(f"wget -q '{info['url']}' -O '{local}'")
    if ret != 0:
        print(f'  {gsm} DOWNLOAD FAILED')
    else:
        print(f'  {gsm} {os.path.getsize(local)/1e6:.0f}MB done')

Checking WIG files...
  GSM429321 downloading (~220MB)...
  GSM429321 221MB done
  GSM429322 downloading (~220MB)...
  GSM429322 221MB done
  GSM432687 downloading (~220MB)...
  GSM432687 233MB done
  GSM432688 downloading (~220MB)...
  GSM432688 233MB done


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — All core functions
# ═══════════════════════════════════════════════════════════════════

# ── WIG parser ────────────────────────────────────────────────────
def parse_wig(filepath, chrom_filter):
    records=[]; current_chrom=None; step=1; fixed=False; pos_counter=1
    with gzip.open(filepath,'rt') as f:
        for line in f:
            line=line.strip()
            if not line or line.startswith('#'): continue
            if line.startswith('fixedStep'):
                parts=dict(p.split('=') for p in line.split()[1:])
                current_chrom=parts.get('chrom','')
                pos_counter=int(parts.get('start',1))
                step=int(parts.get('step',1)); fixed=True
            elif line.startswith('variableStep'):
                parts=dict(p.split('=') for p in line.split()[1:])
                current_chrom=parts.get('chrom',''); fixed=False
            elif line.startswith('track') or line.startswith('browser'): continue
            else:
                if current_chrom!=chrom_filter: continue
                if fixed:
                    try: records.append((pos_counter,float(line))); pos_counter+=step
                    except ValueError: pass
                else:
                    p=line.split()
                    if len(p)==2:
                        try: records.append((int(p[0]),float(p[1])))
                        except ValueError: pass
    if not records: return pd.DataFrame(columns=['pos','beta'])
    df=pd.DataFrame(records,columns=['pos','beta'])
    df=df.drop_duplicates('pos').sort_values('pos').reset_index(drop=True)
    if len(df)>0 and df['beta'].max()>1.5: df['beta']/=100.0
    df['beta']=df['beta'].clip(0.0,1.0).astype(np.float32)
    return df


# ── V6 Hi-C proxy — FIXED ─────────────────────────────────────────
def build_hic_proxy_v6(positions, primary_beta,
                        resolution=25_000, K=5,
                        chunk=500, max_bin_dist=2000):
    """
    V6 Hi-C proxy: fixes all V5 coverage=0% bugs.

    Fixes applied:
      1. NaN bins filled with chromosome-wide mean BEFORE correlation
      2. Zero-variance bins get distance-decay fallback contacts
         (nearest K bins by genomic distance, no correlation filter)
      3. score filter changed from score>0 to top-K by |score|
         (anticorrelated bins = opposite compartments, still informative)
      4. Derived context channels computed AFTER NaN fill
    """
    n_cpgs   = len(positions)
    cpg_bins = (positions // resolution).astype(np.int64)  # int64 safe
    unique_b = np.unique(cpg_bins)
    n_bins   = len(unique_b)
    b2i      = {int(b): i for i, b in enumerate(unique_b)}  # int keys

    # Per-bin mean beta  (n_bins, 4)
    bin_primary = np.zeros((n_bins, 4), dtype=np.float32)
    for idx, b in enumerate(unique_b):
        mask = cpg_bins == b
        vals = primary_beta[mask]
        if len(vals) > 0:
            col_means = np.nanmean(vals, axis=0)
            # Replace remaining NaN with 0.5 (neutral methylation)
            col_means = np.where(np.isnan(col_means), 0.5, col_means)
            bin_primary[idx] = col_means
        else:
            bin_primary[idx] = 0.5  # fallback for empty bins

    # FIX 1: Fill any residual NaN with column mean across all bins
    for col in range(4):
        col_nan = np.isnan(bin_primary[:, col])
        if col_nan.any():
            col_mean = np.nanmean(bin_primary[:, col])
            col_mean = 0.5 if np.isnan(col_mean) else col_mean
            bin_primary[col_nan, col] = col_mean

    # Derived context channels  (n_bins, 4)
    h1_discord    = np.abs(bin_primary[:, 0] - bin_primary[:, 1])
    imr90_discord = np.abs(bin_primary[:, 2] - bin_primary[:, 3])
    h1_var        = np.var(bin_primary[:, :2], axis=1)
    cross_div     = np.abs(bin_primary[:, :2].mean(axis=1) -
                           bin_primary[:, 2:].mean(axis=1))
    ctx = np.stack([h1_discord, imr90_discord, h1_var, cross_div], axis=1)
    # Normalise (safe: avoid division by zero)
    for col in range(4):
        mx = ctx[:, col].max()
        if mx > 1e-8:
            ctx[:, col] /= mx

    # Full 8-column bin feature matrix  (n_bins, 8)
    bin_full = np.concatenate([bin_primary, ctx.astype(np.float32)], axis=1)

    # FIX 2: Identify zero-variance bins (will get fallback contacts)
    bin_std = bin_full.std(axis=1)   # (n_bins,)
    zero_var_bins = set(np.where(bin_std < 1e-6)[0])

    coverage_target = f'{(len(unique_b) - len(zero_var_bins)):,}/{n_bins:,} variable bins'
    print(f'    Bins: {n_bins:,}  |  Variable bins: {coverage_target}')

    # Build top-K contacts
    bin_top_contacts = {}
    for start in range(0, n_bins, chunk):
        end = min(start + chunk, n_bins)
        for i_local, i_global in enumerate(range(start, end)):
            dist_mask = (np.abs(np.arange(n_bins) - i_global) <= max_bin_dist)
            dist_mask[i_global] = False
            cand_idx  = np.where(dist_mask)[0]
            if len(cand_idx) == 0:
                bin_top_contacts[i_global] = []; continue

            # FIX 2: Zero-variance bin → use pure distance-decay fallback
            if i_global in zero_var_bins:
                distances = np.abs(cand_idx - i_global).astype(float)
                scores    = np.exp(-distances / 400)
                top_k     = np.argpartition(scores, -min(K, len(scores)))[-min(K, len(scores)):]
                top_k     = top_k[np.argsort(scores[top_k])[::-1]]
                bin_top_contacts[i_global] = [(cand_idx[j], float(scores[j])) for j in top_k]
                continue

            candidates = bin_full[cand_idx]   # (m, 8)
            my_vec     = bin_full[i_global]   # (8,)
            my_c       = my_vec - my_vec.mean()
            cand_c     = candidates - candidates.mean(axis=1, keepdims=True)
            my_norm    = np.sqrt((my_c**2).sum()) + 1e-8
            cand_norm  = np.sqrt((cand_c**2).sum(axis=1)) + 1e-8
            corr       = (cand_c @ my_c) / (cand_norm * my_norm)
            distances  = np.abs(cand_idx - i_global).astype(float)
            decay      = np.exp(-distances / 400)
            score      = corr * decay

            # FIX 3: sort by |score|, keep top-K (was: filter score>0 → missed negatives)
            abs_score = np.abs(score)
            top_k     = np.argpartition(abs_score, -min(K, len(abs_score)))[-min(K, len(abs_score)):]
            top_k     = top_k[np.argsort(abs_score[top_k])[::-1]]
            bin_top_contacts[i_global] = [
                (cand_idx[j], float(score[j]))   # preserve sign
                for j in top_k if abs_score[j] > 1e-8]

        if start % 2000 == 0 and start > 0:
            print(f'    Hi-C proxy: {start:,}/{n_bins:,} bins...')

    # Assign K×8=40 features per CpG
    hic_feat = np.zeros((n_cpgs, K * 8), dtype=np.float32)
    for cpg_i, my_bin_raw in enumerate(cpg_bins):
        bin_idx  = b2i.get(int(my_bin_raw), -1)
        if bin_idx < 0: continue
        contacts = bin_top_contacts.get(bin_idx, [])
        for j, (partner_idx, score) in enumerate(contacts[:K]):
            hic_feat[cpg_i, j*8:(j+1)*8] = bin_full[partner_idx] * score

    coverage = (hic_feat.sum(axis=1) != 0).mean() * 100
    if coverage < 50:
        print(f'    ⚠️  Coverage still low ({coverage:.1f}%) — check beta matrix for all-NaN columns')
    print(f'    Hi-C V6 coverage: {coverage:.1f}%  shape: {hic_feat.shape}')
    return hic_feat


# ── Masked windows ────────────────────────────────────────────────
def build_windows(beta_m, window_size=21, mask_rate=0.40, stride=3, seed=42):
    rng = np.random.default_rng(seed)
    n_cpgs, n_samples = beta_m.shape
    center = window_size // 2
    n_extra = int(window_size * mask_rate) - 1
    all_X, all_y = [], []
    for s in range(n_samples):
        col  = beta_m[:, s]
        wins = np.lib.stride_tricks.sliding_window_view(col, window_size)[::stride].copy()
        y_s  = wins[:, center].copy()
        wins[:, center] = 0.0
        nb_idx = np.array([i for i in range(window_size) if i != center])
        for w in range(len(wins)):
            wins[w, rng.choice(nb_idx, size=min(n_extra,len(nb_idx)), replace=False)] = 0.0
        all_X.append(wins); all_y.append(y_s)
    return (np.concatenate(all_X).astype(np.float32),
            np.concatenate(all_y).astype(np.float32))


# ── RaM + DMB (V6: z-score normalised threshold) ──────────────────
def ram_null_safe(beta_arr, block_size=500, n_perm=1000, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(beta_arr)
    bs  = min(block_size, max(10, n // 4))
    if n < 40: return np.zeros(n_perm, dtype=np.float32)
    null = []
    for _ in range(n_perm):
        idx   = rng.permutation(n); bp = beta_arr[idx]
        start = rng.integers(0, max(1, n - bs))
        b     = bp[start:start+bs]
        if b.shape[0] < 4: null.append(0.0); continue
        v = abs(b[:, :2].mean() - b[:, 2:4].mean())
        null.append(0.0 if np.isnan(v) else float(v))
    return np.array(null, dtype=np.float32)


def compute_dmbs_safe(pos_arr, beta_arr, err_arr, block_size=500, thresh=0.05):
    records=[]; n=len(pos_arr); bs=min(block_size, max(10, n//4))
    if bs < 10: return pd.DataFrame(columns=[
        'start','end','mid_mb','h1_beta','imr90_beta','delta',
        'abs_delta','direction','mean_recon_err','ram_sig_99'])
    for start in range(0, n - bs, bs // 2):
        end=start+bs; b=beta_arr[start:end]; pos_s=pos_arr[start:end]; e=err_arr[start:end]
        if b.shape[0]<4: continue
        h1=b[:,:2].mean(); im=b[:,2:4].mean(); delta=im-h1
        records.append({'start':int(pos_s[0]),'end':int(pos_s[-1]),
                        'mid_mb':float(pos_s[len(pos_s)//2]/1e6),
                        'h1_beta':float(h1),'imr90_beta':float(im),
                        'delta':float(delta),'abs_delta':float(abs(delta)),
                        'direction':'hyper_IMR90' if delta>0 else 'hypo_IMR90',
                        'mean_recon_err':float(e.mean()),
                        'ram_sig_99':bool(abs(delta)>thresh)})
    return pd.DataFrame(records)


# ── CT score (V6: z-score normalised) ────────────────────────────
def compute_ct_scores_v6(positions, beta_mat, ct_window=50, ct_step=25):
    records=[]; n=len(positions)
    raw_scores=[]
    for start in range(0, n-ct_window, ct_step):
        end=start+ct_window; b=beta_mat[start:end]; pos_s=positions[start:end]
        overall_var   = float(b.var(axis=0).mean())
        mean_beta     = b.mean(axis=1)
        switches      = float((np.abs(np.diff(mean_beta))>0.2).mean())
        h1_discord    = float(np.abs(b[:,0]-b[:,1]).mean())
        imr90_discord = float(np.abs(b[:,2]-b[:,3]).mean())
        ct_score      = 0.40*overall_var+0.30*switches+0.15*h1_discord+0.15*imr90_discord
        raw_scores.append(ct_score)
        records.append({'start':int(pos_s[0]),'end':int(pos_s[-1]),
                        'mid_mb':float(pos_s[len(pos_s)//2]/1e6),
                        'ct_score_raw':ct_score,
                        'overall_var':overall_var,'switch_rate':switches,
                        'h1_discord':h1_discord,'imr90_discord':imr90_discord,
                        'h1_mean':float(b[:,:2].mean()),'imr90_mean':float(b[:,2:4].mean()),
                        'delta':float(b[:,2:4].mean()-b[:,:2].mean())})
    if not records:
        return pd.DataFrame()
    ct_df = pd.DataFrame(records)
    # V6 FIX: z-score normalise so thresholds are meaningful across chromosomes
    mu    = ct_df['ct_score_raw'].mean()
    sigma = ct_df['ct_score_raw'].std() + 1e-8
    ct_df['ct_score']  = (ct_df['ct_score_raw'] - mu) / sigma  # z-score
    ct_df['ct_sig_99'] = ct_df['ct_score'] > 2.326   # one-tailed p<0.01 z-threshold
    ct_df['ct_sig_95'] = ct_df['ct_score'] > 1.645   # one-tailed p<0.05
    return ct_df


def is_done(chrom):
    d = f'{RESULTS_DIR}/{chrom}'
    return all(os.path.exists(f'{d}/{chrom}_V6_{x}') for x in
               ['metrics.json','y_true.npy','dmb_p.csv'])

print('All functions defined.')

All functions defined.


In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Meth3D-Net V6 model
# Same architecture as V5 but with hic_dim=40 from the fixed proxy
# ═══════════════════════════════════════════════════════════════════

class GraphConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin=nn.Linear(in_dim,out_dim); self.norm=nn.LayerNorm(out_dim)
    def forward(self,x,adj):
        h=torch.bmm(adj.unsqueeze(0).expand(x.size(0),-1,-1),x)
        return F.gelu(self.norm(self.lin(h)))

class HiCEncoder_V6(nn.Module):
    """3-layer encoder with LAR gates. hic_dim=40 from V6 proxy."""
    def __init__(self, hic_dim=40, out_dim=64):
        super().__init__()
        self.fc1=nn.Linear(hic_dim,128); self.fc2=nn.Linear(128,96); self.fc3=nn.Linear(96,out_dim)
        self.n1=nn.LayerNorm(128); self.n2=nn.LayerNorm(96); self.n3=nn.LayerNorm(out_dim)
        self.drop=nn.Dropout(0.1)
        self.gate1=nn.Parameter(torch.ones(128)); self.gate2=nn.Parameter(torch.ones(96))
    def forward(self,x):
        h=F.gelu(self.n1(self.fc1(x)))*torch.sigmoid(self.gate1); h=self.drop(h)
        h=F.gelu(self.n2(self.fc2(h)))*torch.sigmoid(self.gate2); h=self.drop(h)
        return self.n3(F.gelu(self.fc3(h)))

class Meth3DNet_V6(nn.Module):
    def __init__(self, window_size=21, hidden=64, gcn_hidden=32,
                 n_gcn_layers=2, hic_dim=40, hic_out=64, pos_dim=8):
        super().__init__()
        self.window_size=window_size; self.pos_dim=pos_dim; center=window_size//2
        self.conv1=nn.Conv1d(1,32,3,padding=1); self.conv2=nn.Conv1d(32,64,3,padding=1)
        self.conv3=nn.Conv1d(64,hidden,3,padding=1)
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.bn1=nn.BatchNorm1d(32); self.bn2=nn.BatchNorm1d(64)
        self.node_embed=nn.Linear(1,gcn_hidden)
        self.gcn_layers=nn.ModuleList([GraphConvLayer(gcn_hidden,gcn_hidden)
                                        for _ in range(n_gcn_layers)])
        self.center_idx=center
        A=torch.zeros(window_size,window_size)
        for i in range(window_size):
            A[i,i]=1.0
            if i>0: A[i,i-1]=0.5
            if i<window_size-1: A[i,i+1]=0.5
        self.register_buffer('adj',A/A.sum(dim=1,keepdim=True))
        self.hic_encoder=HiCEncoder_V6(hic_dim,hic_out)
        fusion_in=hidden+gcn_hidden+hic_out+pos_dim  # 64+32+64+8=168
        self.fc1=nn.Linear(fusion_in,256); self.fc2=nn.Linear(256,128); self.fc3=nn.Linear(128,1)
        self.drop=nn.Dropout(0.2)
    def sinusoidal_pos(self,positions):
        B=positions.size(0); d=self.pos_dim
        pos=positions.float().unsqueeze(1)/1e6
        div=torch.exp(torch.arange(0,d,2).float()*(-np.log(10000.)/d)).to(positions.device)
        pe=torch.zeros(B,d).to(positions.device)
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        return pe
    def forward(self,x,hic,pos):
        h=F.gelu(self.bn1(self.conv1(x.unsqueeze(1))))
        h=F.gelu(self.bn2(self.conv2(h)))
        h=F.gelu(self.conv3(h)); h=self.pool(h).squeeze(-1)
        g=F.gelu(self.node_embed(x.unsqueeze(-1)))
        for layer in self.gcn_layers: g=layer(g,self.adj)
        g=g[:,self.center_idx,:]
        hi=self.hic_encoder(hic); pe=self.sinusoidal_pos(pos)
        h=torch.cat([h,g,hi,pe],dim=1)
        h=F.gelu(self.fc1(h)); h=self.drop(h); h=F.gelu(self.fc2(h))
        return torch.sigmoid(self.fc3(h)).squeeze(-1)

_m=Meth3DNet_V6(hic_dim=HIC_DIM,hic_out=HIC_OUT)
print(f'Meth3D-Net V6: {sum(p.numel() for p in _m.parameters()):,} parameters'); del _m

Meth3D-Net V6: 122,113 parameters


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Main training loop (all chromosomes)
# ═══════════════════════════════════════════════════════════════════
done      = [c for c in CHROMS_ALL if is_done(c)]
remaining = [c for c in CHROMS_ALL if c not in done]
print(f'Status: {len(done)}/24 done  |  Remaining: {remaining}')

genome_metrics = []

for CHROM in CHROMS_ALL:
    CHROM_DIR  = f'{RESULTS_DIR}/{CHROM}'
    os.makedirs(CHROM_DIR, exist_ok=True)
    IS_CHRY    = (CHROM == 'chrY')
    CENTROMERE = CENTROMERES.get(CHROM, 0)
    GENES      = HOTSPOTS.get(CHROM, {})

    if is_done(CHROM):
        with open(f'{CHROM_DIR}/{CHROM}_V6_metrics.json') as f:
            m = json.load(f)
        genome_metrics.append(m)
        print(f'{CHROM}  already done — r={m["pearson"]:.4f}')
        continue

    print(f'\n{"="*55}  {CHROM}  {"="*5}')

    # ── Parse ──────────────────────────────────────────────────────
    print(f'  [1/6] Parsing {CHROM}...')
    sample_dfs = {}
    for gsm in PRIMARY_GSMS:
        info = SAMPLES[gsm]
        if not os.path.exists(info['local']): continue
        df = parse_wig(info['local'], CHROM)
        sample_dfs[gsm] = df
        print(f'    {gsm}  {len(df):,} CpGs')

    base = sample_dfs[PRIMARY_GSMS[0]][['pos','beta']].rename(columns={'beta':PRIMARY_GSMS[0]})
    for gsm in PRIMARY_GSMS[1:]:
        df_gsm = sample_dfs.get(gsm, pd.DataFrame(columns=['pos','beta']))
        if len(df_gsm)==0: base[gsm]=np.nan; continue
        base = base.merge(df_gsm[['pos','beta']].rename(columns={'beta':gsm}),
                          on='pos', how='outer')
    base = base.sort_values('pos').reset_index(drop=True)
    positions    = base['pos'].values
    all_mat      = base[PRIMARY_GSMS].values.astype(np.float32)
    col_means    = np.nanmean(all_mat, axis=0, keepdims=True)
    col_means    = np.where(np.isnan(col_means), 0.5, col_means)
    nan_mask     = np.isnan(all_mat)
    all_mat      = np.where(nan_mask, np.broadcast_to(col_means, all_mat.shape), all_mat)
    primary_beta = all_mat
    print(f'    Positions: {len(positions):,}  |  NaN rate: {nan_mask.mean()*100:.1f}%')

    # ── Hi-C proxy V6 ──────────────────────────────────────────────
    print(f'  [2/6] Building V6 Hi-C proxy...')
    hic_feat = build_hic_proxy_v6(positions, primary_beta, resolution=RESOLUTION, K=HIC_K)

    # ── Windows ────────────────────────────────────────────────────
    print(f'  [3/6] Building windows...')
    X_all, y_all = build_windows(primary_beta)
    n_win = (len(positions) - WINDOW_SIZE) // STRIDE + 1
    center_idx   = np.arange(CENTER, CENTER + n_win*STRIDE, STRIDE)[:n_win]
    center_idx_all = np.tile(center_idx, 4)
    hic_for_win    = hic_feat[center_idx_all]
    pos_for_win    = positions[center_idx_all]
    h1_sl=slice(0,2*n_win); im_sl=slice(2*n_win,4*n_win)
    X_tr=torch.tensor(X_all[h1_sl],dtype=torch.float32)
    y_tr=torch.tensor(y_all[h1_sl],dtype=torch.float32)
    h_tr=torch.tensor(hic_for_win[h1_sl],dtype=torch.float32)
    p_tr=torch.tensor(pos_for_win[h1_sl],dtype=torch.long)
    X_te=torch.tensor(X_all[im_sl],dtype=torch.float32)
    y_te=torch.tensor(y_all[im_sl],dtype=torch.float32)
    h_te=torch.tensor(hic_for_win[im_sl],dtype=torch.float32)
    p_te=torch.tensor(pos_for_win[im_sl],dtype=torch.long)
    del X_all, y_all; gc.collect()
    n_val=int(0.1*len(X_tr)); n_train=len(X_tr)-n_val
    train_ds,val_ds=random_split(TensorDataset(X_tr,y_tr,h_tr,p_tr),[n_train,n_val],
                                  generator=torch.Generator().manual_seed(42))
    test_ds=TensorDataset(X_te,y_te,h_te,p_te)
    TL=DataLoader(train_ds,BATCH_SIZE,shuffle=True,num_workers=0)
    VL=DataLoader(val_ds,BATCH_SIZE,shuffle=False,num_workers=0)
    TeL=DataLoader(test_ds,BATCH_SIZE,shuffle=False,num_workers=0)
    print(f'    Train:{n_train:,}  Val:{n_val:,}  Test:{len(X_te):,}')

    # ── Train ──────────────────────────────────────────────────────
    print(f'  [4/6] Training V6...')
    model     = Meth3DNet_V6(hic_dim=HIC_DIM,hic_out=HIC_OUT).to(DEVICE)
    optimizer = optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,'min',patience=2,factor=0.5)
    loss_fn   = nn.MSELoss(); best_val=float('inf'); history={'train':[],'val':[]}
    model_path = f'{CHROM_DIR}/{CHROM}_V6_model_best.pt'
    print(f'  {"Ep":>3}  {"Train":>10}  {"Val":>10}  {"LR":>8}')
    for epoch in range(1, EPOCHS+1):
        model.train(); tr=0.0
        for xb,yb,hb,pb in TL:
            xb,yb,hb,pb=xb.to(DEVICE),yb.to(DEVICE),hb.to(DEVICE),pb.to(DEVICE)
            optimizer.zero_grad(); loss=loss_fn(model(xb,hb,pb),yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step(); tr+=loss.item()
        tr/=len(TL); model.eval(); vl=0.0
        with torch.no_grad():
            for xb,yb,hb,pb in VL:
                xb,yb,hb,pb=xb.to(DEVICE),yb.to(DEVICE),hb.to(DEVICE),pb.to(DEVICE)
                vl+=loss_fn(model(xb,hb,pb),yb).item()
        vl/=len(VL); scheduler.step(vl); lr=optimizer.param_groups[0]['lr']
        history['train'].append(tr); history['val'].append(vl)
        if vl<best_val: best_val=vl; torch.save(model.state_dict(),model_path)
        if epoch%5==0 or vl<best_val:
            print(f'  {epoch:>3}  {tr:>10.6f}  {vl:>10.6f}  {lr:>8.2e}{" ✓" if vl==best_val else ""}')
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
        gc.collect()

    # ── Evaluate ───────────────────────────────────────────────────
    model.load_state_dict(torch.load(model_path,map_location=DEVICE)); model.eval()
    preds,trues=[],[]
    with torch.no_grad():
        for xb,yb,hb,pb in TeL:
            xb,hb,pb=xb.to(DEVICE),hb.to(DEVICE),pb.to(DEVICE)
            preds.append(model(xb,hb,pb).cpu().numpy()); trues.append(yb.numpy())
    y_pred=np.concatenate(preds); y_true=np.concatenate(trues)
    r_p=float(pearsonr(y_true,y_pred)[0])
    r_s=float(spearmanr(y_true,y_pred)[0])
    mse=float(np.mean((y_true-y_pred)**2))
    mae=float(np.mean(np.abs(y_true-y_pred)))
    r2 =float(1-np.sum((y_true-y_pred)**2)/np.sum((y_true-y_true.mean())**2))
    print(f'  Result: r={r_p:.4f}  R²={r2:.4f}  MSE={mse:.6f}  MAE={mae:.6f}')
    np.save(f'{CHROM_DIR}/{CHROM}_V6_y_true.npy',y_true)
    np.save(f'{CHROM_DIR}/{CHROM}_V6_y_pred.npy',y_pred)
    metrics={'chrom':CHROM,'version':'V6','pearson':r_p,'spearman':r_s,
             'r2':r2,'mse':mse,'mae':mae,'best_val_mse':float(best_val),
             'n_cpgs':int(len(positions)),'hic_coverage':float((hic_feat.sum(1)!=0).mean()*100),
             'train_history':[float(x) for x in history['train']],
             'val_history':[float(x) for x in history['val']]}
    with open(f'{CHROM_DIR}/{CHROM}_V6_metrics.json','w') as f: json.dump(metrics,f,indent=2)

    # ── DMB analysis ───────────────────────────────────────────────
    print(f'  [5/6] DMB + CT analysis...')
    n_per=len(y_true)//2
    err_mean=(np.abs(y_true[:n_per]-y_pred[:n_per])+np.abs(y_true[n_per:]-y_pred[n_per:]))/2
    err_per_cpg=np.zeros(len(positions),dtype=np.float32)
    cnt=np.zeros(len(positions),dtype=np.int32)
    for wi,ci in enumerate(center_idx):
        if wi<len(err_mean): err_per_cpg[ci]+=err_mean[wi]; cnt[ci]+=1
    m=cnt>0; err_per_cpg[m]/=cnt[m]

    BLOCK=500; p_mask=positions<CENTROMERE; q_mask=positions>=CENTROMERE
    null_p=ram_null_safe(primary_beta[p_mask],BLOCK)
    null_q=ram_null_safe(primary_beta[q_mask],BLOCK)
    tp=np.percentile(null_p,99) if null_p.sum()>0 else 0.05
    tq=np.percentile(null_q,99) if null_q.sum()>0 else 0.05

    if not IS_CHRY:
        dmb_p=compute_dmbs_safe(positions[p_mask],primary_beta[p_mask],err_per_cpg[p_mask],BLOCK,tp)
        dmb_q=compute_dmbs_safe(positions[q_mask],primary_beta[q_mask],err_per_cpg[q_mask],BLOCK,tq)
    else:
        dmb_p=pd.DataFrame(); dmb_q=pd.DataFrame()

    dmb_p.to_csv(f'{CHROM_DIR}/{CHROM}_V6_dmb_p.csv',index=False)
    dmb_q.to_csv(f'{CHROM_DIR}/{CHROM}_V6_dmb_q.csv',index=False)
    sig_p=len(dmb_p[dmb_p.ram_sig_99]) if len(dmb_p) else 0
    sig_q=len(dmb_q[dmb_q.ram_sig_99]) if len(dmb_q) else 0
    print(f'    DMBp sig:{sig_p}  DMBq sig:{sig_q}')

    ct_df=compute_ct_scores_v6(positions,primary_beta)
    ct_df.to_csv(f'{CHROM_DIR}/{CHROM}_V6_ct_scores.csv',index=False)
    ct_sig=ct_df['ct_sig_99'].sum() if len(ct_df) else 0
    print(f'    CT-sig (z>2.33): {ct_sig:,} windows')

    metrics['dmb_p_sig']=sig_p; metrics['dmb_q_sig']=sig_q; metrics['ct_sig_99']=int(ct_sig)
    with open(f'{CHROM_DIR}/{CHROM}_V6_metrics.json','w') as f: json.dump(metrics,f,indent=2)
    genome_metrics.append(metrics)

    del model,X_tr,X_te,y_tr,y_te,h_tr,h_te,p_tr,p_te
    del primary_beta,hic_feat,all_mat,train_ds,val_ds,test_ds
    gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    print(f'  ✅ {CHROM} complete')

print(f'\n{"="*55}\n  ALL CHROMOSOMES COMPLETE\n{"="*55}')

Status: 0/24 done  |  Remaining: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX', 'chrY']

=======================================================  chr1  =====
  [1/6] Parsing chr1...
    GSM429321  4,233,942 CpGs
    GSM429322  4,233,942 CpGs
    GSM432687  4,321,280 CpGs
    GSM432688  4,321,280 CpGs
    Positions: 4,347,676  |  NaN rate: 1.6%
  [2/6] Building V6 Hi-C proxy...
    Bins: 9,035  |  Variable bins: 9,035/9,035 variable bins
    Hi-C proxy: 2,000/9,035 bins...
    Hi-C proxy: 4,000/9,035 bins...
    Hi-C proxy: 6,000/9,035 bins...
    Hi-C proxy: 8,000/9,035 bins...
    Hi-C V6 coverage: 100.0%  shape: (4347676, 40)
  [3/6] Building windows...
    Train:2,608,595  Val:289,843  Test:2,898,438
  [4/6] Training V6...
   Ep       Train         Val        LR
    5    0.012958    0.012880  1.00e-03 ✓
   10    0.012457    0.012405  

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Genome-wide summary CSV
# ═══════════════════════════════════════════════════════════════════
all_m=[]
for CHROM in CHROMS_ALL:
    mp=f'{RESULTS_DIR}/{CHROM}/{CHROM}_V6_metrics.json'
    if os.path.exists(mp):
        with open(mp) as f: all_m.append(json.load(f))

sum_df=pd.DataFrame(all_m)[['chrom','n_cpgs','pearson','spearman','r2','mse','mae',
                              'hic_coverage','dmb_p_sig','dmb_q_sig','ct_sig_99']]
sum_df.to_csv(f'{RESULTS_DIR}/V6_genome_summary.csv',index=False)
print('Genome-wide summary:')
print(sum_df.to_string(index=False,float_format='%.4f'))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — FIGURE 1: Genome-wide Prediction Accuracy
# Four panels: Pearson r, R², MAE, Hi-C coverage
# ═══════════════════════════════════════════════════════════════════
print('Generating Figure 1: Genome-wide prediction accuracy...')

df = sum_df.copy()
chroms = df['chrom'].tolist()
x = np.arange(len(chroms))

# Colour: autosomes = purple gradient, chrX = blue, chrY = grey
def chrom_color(c):
    if c == 'chrY': return '#AAAAAA'
    if c == 'chrX': return '#2166AC'
    n = int(c.replace('chr',''))
    return plt.cm.plasma(0.3 + 0.4 * (n-1)/21)

colors = [chrom_color(c) for c in chroms]

fig, axes = plt.subplots(2, 2, figsize=(20, 10))
fig.patch.set_facecolor('#F8F9FA')

# ── Panel A: Pearson r ───────────────────────────────────────────
ax = axes[0,0]
bars = ax.bar(x, df['pearson'], color=colors, alpha=0.85, edgecolor='white', width=0.75)
ax.axhline(0.85, color='#D6604D', lw=2, ls='--', label='r = 0.85')
ax.axhline(0.90, color='#F4A582', lw=1.5, ls=':', label='r = 0.90')
ax.set_xticks(x); ax.set_xticklabels(chroms, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1.0); ax.set_ylabel('Pearson r', fontsize=11)
ax.set_title('A  Methylation Prediction Accuracy (V6)', fontsize=11, fontweight='bold', loc='left')
ax.legend(fontsize=9, frameon=False)
ax.spines[['top','right']].set_visible(False)
# Annotate top performers
for i, (c, r) in enumerate(zip(chroms, df['pearson'])):
    if r > 0.90:
        ax.text(i, r + 0.005, f'{r:.3f}', ha='center', fontsize=7, color='#333333', fontweight='bold')

# ── Panel B: R² ──────────────────────────────────────────────────
ax = axes[0,1]
ax.bar(x, df['r2'], color=colors, alpha=0.85, edgecolor='white', width=0.75)
ax.set_xticks(x); ax.set_xticklabels(chroms, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 1.0); ax.set_ylabel('R²', fontsize=11)
ax.set_title('B  Coefficient of Determination', fontsize=11, fontweight='bold', loc='left')
ax.spines[['top','right']].set_visible(False)

# ── Panel C: MAE ─────────────────────────────────────────────────
ax = axes[1,0]
ax.bar(x, df['mae'], color=colors, alpha=0.85, edgecolor='white', width=0.75)
ax.set_xticks(x); ax.set_xticklabels(chroms, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean Absolute Error', fontsize=11)
ax.set_title('C  Mean Absolute Error', fontsize=11, fontweight='bold', loc='left')
ax.spines[['top','right']].set_visible(False)

# ── Panel D: Hi-C V6 coverage ────────────────────────────────────
ax = axes[1,1]
cov_colors = ['#1A7A1A' if v >= 80 else '#D6604D' if v < 50 else '#F4A582'
              for v in df['hic_coverage']]
ax.bar(x, df['hic_coverage'], color=cov_colors, alpha=0.85, edgecolor='white', width=0.75)
ax.axhline(80, color='#1A7A1A', lw=2, ls='--', label='80% threshold')
ax.axhline(50, color='#D6604D', lw=1.5, ls=':', label='50% warning')
ax.set_xticks(x); ax.set_xticklabels(chroms, rotation=45, ha='right', fontsize=8)
ax.set_ylim(0, 105); ax.set_ylabel('Hi-C Proxy Coverage (%)', fontsize=11)
ax.set_title('D  V6 Hi-C Proxy Coverage  (V5 was 0% — now fixed)', 
             fontsize=11, fontweight='bold', loc='left')
ax.legend(fontsize=9, frameon=False)
ax.spines[['top','right']].set_visible(False)

# Legend patches for chromosome groups
legend_handles = [
    mpatches.Patch(color=plt.cm.plasma(0.4), label='Autosomes (chr1–22)'),
    mpatches.Patch(color='#2166AC', label='chrX'),
    mpatches.Patch(color='#AAAAAA', label='chrY (H1-only)'),
]
fig.legend(handles=legend_handles, fontsize=9, frameon=False,
           loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.01))

fig.suptitle('Meth3D-Net V6 — Genome-Wide Prediction Performance\n'
             'H1 ESC vs IMR90 | WGBS GSE19418 | '
             f'Mean Pearson r = {df[df.chrom!="chrY"]["pearson"].mean():.3f}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig1_pdf = f'{RESULTS_DIR}/Fig1_Genome_Accuracy.pdf'
fig1_png = f'{RESULTS_DIR}/Fig1_Genome_Accuracy.png'
plt.savefig(fig1_pdf, dpi=300, bbox_inches='tight')
plt.savefig(fig1_png, dpi=150, bbox_inches='tight')
plt.close()
print(f'Figure 1 saved: {fig1_pdf}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — FIGURE 2: Genome-wide Instability Landscape
# Three-track plot: DMB density | CT instability | combined heatmap
# ═══════════════════════════════════════════════════════════════════
print('Generating Figure 2: Genome-wide instability landscape...')

CHROM_ORDER = [c for c in CHROMS_ALL if c != 'chrY' and is_done(c)]
CHROM_SIZES_HG19 = {
    'chr1':249_250_621,'chr2':243_199_373,'chr3':198_022_430,'chr4':191_154_276,
    'chr5':180_915_260,'chr6':171_115_067,'chr7':159_138_663,'chr8':146_364_022,
    'chr9':141_213_431,'chr10':135_534_747,'chr11':135_006_516,'chr12':133_851_895,
    'chr13':115_169_878,'chr14':107_349_540,'chr15':102_531_392,'chr16':90_354_753,
    'chr17':81_195_210,'chr18':78_077_248,'chr19':59_128_983,'chr20':63_025_520,
    'chr21':48_129_895,'chr22':51_304_566,'chrX':155_270_560,
}

# Build cumulative offsets
offsets = {}; offset = 0
for c in CHROM_ORDER:
    offsets[c] = offset
    offset += CHROM_SIZES_HG19.get(c, 250_000_000)
GENOME_SIZE = offset

# Load DMB and CT data
WINDOW_5MB = 5_000_000
genome_bins = int(np.ceil(GENOME_SIZE / WINDOW_5MB))
dmb_p_arr = np.zeros(genome_bins);  dmb_q_arr = np.zeros(genome_bins)
ct_arr    = np.zeros(genome_bins)

for CHROM in CHROM_ORDER:
    cdir = f'{RESULTS_DIR}/{CHROM}'
    off  = offsets[CHROM]

    for arm, arr in [('p', dmb_p_arr), ('q', dmb_q_arr)]:
        fp = f'{cdir}/{CHROM}_V6_dmb_{arm}.csv'
        if not os.path.exists(fp): continue
        df_dmb = pd.read_csv(fp)
        df_sig = df_dmb[df_dmb.get('ram_sig_99', pd.Series([False]*len(df_dmb)))==True]
        for _, row in df_sig.iterrows():
            bin_idx = int((off + row['start']) // WINDOW_5MB)
            if 0 <= bin_idx < genome_bins:
                arr[bin_idx] += abs(row.get('delta', 0.0))

    ct_fp = f'{cdir}/{CHROM}_V6_ct_scores.csv'
    if os.path.exists(ct_fp):
        df_ct = pd.read_csv(ct_fp)
        for _, row in df_ct.iterrows():
            bin_idx = int((off + row['start']) // WINDOW_5MB)
            if 0 <= bin_idx < genome_bins:
                ct_arr[bin_idx] = max(ct_arr[bin_idx], row.get('ct_score', 0.0))

bin_x = np.arange(genome_bins) * WINDOW_5MB / 1e6

fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor('#F8F9FA')
gs  = gridspec.GridSpec(3, 1, figure=fig, hspace=0.45, top=0.93, bottom=0.08)

# ── Panel A: DMB density ─────────────────────────────────────────
ax_A = fig.add_subplot(gs[0])
ax_A.fill_between(bin_x, dmb_q_arr, alpha=0.7, color='#4393C3', label='q-arm Σ|Δβ|')
ax_A.fill_between(bin_x, -dmb_p_arr, alpha=0.7, color='#D6604D', label='p-arm Σ|Δβ| (inverted)')
ax_A.axhline(0, color='black', lw=0.8)
# Chromosome boundaries
for c in CHROM_ORDER:
    bnd = offsets[c] / 1e6
    ax_A.axvline(bnd, color='#BBBBBB', lw=0.7, alpha=0.8)
    midpt = (offsets[c] + CHROM_SIZES_HG19.get(c,250_000_000)/2) / 1e6
    ax_A.text(midpt, ax_A.get_ylim()[1]*0.85 if ax_A.get_ylim()[1] else 0.05,
              c.replace('chr',''), fontsize=6.5, ha='center', color='#555555', fontweight='bold')
ax_A.set_ylabel('Σ|Δβ| (DMB magnitude)', fontsize=11)
ax_A.set_title('A  RaM-Significant Differentially Methylated Block Density — All Chromosomes',
               fontsize=11, fontweight='bold', loc='left')
ax_A.legend(fontsize=9, frameon=False, loc='upper right')
ax_A.spines[['top','right']].set_visible(False)
ax_A.tick_params(labelbottom=False)

# ── Panel B: CT score track ──────────────────────────────────────
ax_B = fig.add_subplot(gs[1], sharex=ax_A)
ct_smooth = uniform_filter1d(ct_arr, max(3, len(ct_arr)//500))
ct_thresh = np.percentile(ct_smooth[ct_smooth>0], 95) if (ct_smooth>0).any() else 1.0
ax_B.fill_between(bin_x, ct_smooth, alpha=0.75, color='#8B0000', label='CT instability (z-score)')
ax_B.axhline(ct_thresh, color='#FF4444', lw=1.5, ls='--', label=f'95th pct ({ct_thresh:.2f})')
ax_B.axhline(2.326, color='#FF8800', lw=1.2, ls=':', label='p<0.01 (z=2.33)')
for c in CHROM_ORDER:
    ax_B.axvline(offsets[c]/1e6, color='#BBBBBB', lw=0.7, alpha=0.8)
ax_B.set_ylabel('CT Score (z-score)', fontsize=11)
ax_B.set_title('B  Chromothripsis Instability Score (V6 z-normalised)',
               fontsize=11, fontweight='bold', loc='left')
ax_B.legend(fontsize=9, frameon=False, loc='upper right')
ax_B.spines[['top','right']].set_visible(False)
ax_B.tick_params(labelbottom=False)

# ── Panel C: Combined CERS-like heatmap ──────────────────────────
ax_C = fig.add_subplot(gs[2], sharex=ax_A)
combined = (dmb_q_arr + dmb_p_arr) * 0.5 + np.clip(ct_smooth, 0, None) * 0.5
combined_norm = combined / (combined.max() + 1e-8)
# Draw as scatter coloured by score
sc = ax_C.scatter(bin_x, np.ones_like(bin_x), c=combined_norm,
                  cmap='YlOrRd', s=40, marker='|', linewidths=3,
                  vmin=0, vmax=1)
plt.colorbar(sc, ax=ax_C, orientation='horizontal', pad=0.25,
             label='Combined instability score (normalised)', shrink=0.4)
for c in CHROM_ORDER:
    ax_C.axvline(offsets[c]/1e6, color='#BBBBBB', lw=0.7, alpha=0.8)
    midpt = (offsets[c] + CHROM_SIZES_HG19.get(c,250_000_000)/2) / 1e6
    ax_C.text(midpt, 1.3, c.replace('chr',''), fontsize=6.5, ha='center',
              color='#555555', fontweight='bold')
ax_C.set_ylim(0.5, 1.5); ax_C.set_yticks([])
ax_C.set_xlabel('Genome position (Mb)', fontsize=11)
ax_C.set_title('C  Combined Epigenetic Instability Map (DMB + CT)',
               fontsize=11, fontweight='bold', loc='left')
ax_C.spines[['top','right','left']].set_visible(False)

fig.suptitle('Meth3D-Net V6 — Genome-Wide Epigenetic Instability Landscape\n'
             'H1 ESC vs IMR90 | WGBS GSE19418 | V6 Hi-C proxy (NaN-fixed, abs-corr)',
             fontsize=12, fontweight='bold', y=0.97)

fig2_pdf = f'{RESULTS_DIR}/Fig2_Instability_Landscape.pdf'
fig2_png = f'{RESULTS_DIR}/Fig2_Instability_Landscape.png'
plt.savefig(fig2_pdf, dpi=300, bbox_inches='tight')
plt.savefig(fig2_png, dpi=150, bbox_inches='tight')
plt.close()
print(f'Figure 2 saved: {fig2_pdf}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — FIGURE 3: HLA Locus Zoom (chr6 25–35 Mb)
# Five panels: methylation tracks | DMBs | CT score |
#              co-methylation contact score | gene annotations
# ═══════════════════════════════════════════════════════════════════
print('Generating Figure 3: HLA locus zoom...')

ZOOM_CHROM  = 'chr6'
ZOOM_START  = 25_000_000
ZOOM_END    = 35_000_000
ZOOM_GENES  = {
    'HLA-A' : (29_910_000, 29_914_000),
    'HLA-B' : (31_321_000, 31_325_000),
    'HLA-C' : (31_236_000, 31_240_000),
    'HLA-DRB1': (32_546_000, 32_557_000),
    'HLA-DQB1': (32_627_000, 32_634_000),
    'TAP1'  : (32_812_000, 32_821_000),
    'TAP2'  : (32_821_000, 32_836_000),
    'PRDM1' : (106_522_000,106_600_000),  # outside zoom — skip if not in range
}
ZOOM_GENES_IN = {g: (s,e) for g,(s,e) in ZOOM_GENES.items()
                 if s >= ZOOM_START and e <= ZOOM_END}

CHROM_DIR_6 = f'{RESULTS_DIR}/{ZOOM_CHROM}'

# Re-parse chr6 methylation (only the zoom region — fast)
print(f'  Parsing {ZOOM_CHROM} for HLA zoom ({ZOOM_START/1e6:.0f}–{ZOOM_END/1e6:.0f} Mb)...')
dfs = {}
for gsm in PRIMARY_GSMS:
    info = SAMPLES[gsm]
    if not os.path.exists(info['local']): continue
    df = parse_wig(info['local'], ZOOM_CHROM)
    dfs[gsm] = df[(df['pos'] >= ZOOM_START) & (df['pos'] <= ZOOM_END)].copy()
    print(f'    {gsm}: {len(dfs[gsm]):,} CpGs in zoom region')

# Merge on shared positions
base6 = dfs[PRIMARY_GSMS[0]][['pos','beta']].rename(columns={'beta':PRIMARY_GSMS[0]})
for gsm in PRIMARY_GSMS[1:]:
    if gsm in dfs:
        base6 = base6.merge(dfs[gsm][['pos','beta']].rename(columns={'beta':gsm}),
                            on='pos', how='outer')
base6 = base6.sort_values('pos').reset_index(drop=True)
pos6  = base6['pos'].values
mat6  = base6[PRIMARY_GSMS].fillna(base6[PRIMARY_GSMS].mean()).values.astype(np.float32)

ROLL = max(10, len(pos6)//500)
h1_smooth  = uniform_filter1d(mat6[:,:2].mean(axis=1), ROLL)
imr_smooth = uniform_filter1d(mat6[:,2:4].mean(axis=1), ROLL)
delta_sm   = uniform_filter1d(mat6[:,2:4].mean(axis=1)-mat6[:,:2].mean(axis=1), ROLL)
pos6_mb    = pos6 / 1e6

# Load DMB and CT for chr6 zoom region
dmb6_p = dmb6_q = pd.DataFrame()
for arm in ['p','q']:
    fp = f'{CHROM_DIR_6}/{ZOOM_CHROM}_V6_dmb_{arm}.csv'
    if os.path.exists(fp):
        df_arm = pd.read_csv(fp)
        df_arm = df_arm[(df_arm['start']>=ZOOM_START) & (df_arm['end']<=ZOOM_END)]
        if arm=='p': dmb6_p=df_arm
        else:        dmb6_q=df_arm

ct6 = pd.DataFrame()
fp_ct = f'{CHROM_DIR_6}/{ZOOM_CHROM}_V6_ct_scores.csv'
if os.path.exists(fp_ct):
    ct_all = pd.read_csv(fp_ct)
    ct6    = ct_all[(ct_all['start']>=ZOOM_START) & (ct_all['end']<=ZOOM_END)]

# Build zoom figure
C_H1='#2166AC'; C_IMR='#D6604D'; C_GENE='#FFD700'
gene_colors=['#FFD700','#FF8C00','#9370DB','#20B2AA','#32CD32','#FF69B4','#87CEEB']

fig = plt.figure(figsize=(20, 18))
fig.patch.set_facecolor('#F8F9FA')
gs  = gridspec.GridSpec(4, 1, figure=fig, hspace=0.50, top=0.93, bottom=0.06)

def add_gene_spans(ax, y_top, genes_dict, gcolors):
    for (gene,(gs_,ge_)), col in zip(genes_dict.items(), gcolors):
        ax.axvspan(gs_/1e6, ge_/1e6, alpha=0.25, color=col, label=gene)
        ax.text((gs_+ge_)/2e6, y_top*0.95, gene, fontsize=7.5,
                ha='center', rotation=30, color=col, fontweight='bold')

# ── Panel A: Methylation landscape ───────────────────────────────
ax_A = fig.add_subplot(gs[0])
ax_A.fill_between(pos6_mb, h1_smooth,  alpha=0.5, color=C_H1,  label='H1 ESC')
ax_A.fill_between(pos6_mb, imr_smooth, alpha=0.45, color=C_IMR, label='IMR90')
ax_A.axvline(CENTROMERES[ZOOM_CHROM]/1e6, color='black', lw=2, ls='--', alpha=0.4,
              label='Centromere' if CENTROMERES[ZOOM_CHROM]>ZOOM_START else '')
add_gene_spans(ax_A, 1.0, ZOOM_GENES_IN, gene_colors)
ax_A.set_ylim(0, 1.05)
ax_A.set_ylabel('Methylation β', fontsize=11)
ax_A.set_title(f'A  HLA Locus Methylation Landscape — chr6 {ZOOM_START/1e6:.0f}–{ZOOM_END/1e6:.0f} Mb',
               fontsize=11, fontweight='bold', loc='left')
ax_A.legend(fontsize=8, frameon=False, ncol=5, loc='lower right')
ax_A.spines[['top','right']].set_visible(False)
ax_A.tick_params(labelbottom=False)

# ── Panel B: Differential methylation Δβ ─────────────────────────
ax_B = fig.add_subplot(gs[1], sharex=ax_A)
ax_B.fill_between(pos6_mb, delta_sm, 0,
                   where=delta_sm>0, alpha=0.7, color=C_IMR, label='IMR90 > H1')
ax_B.fill_between(pos6_mb, delta_sm, 0,
                   where=delta_sm<=0, alpha=0.7, color=C_H1, label='H1 > IMR90')
ax_B.axhline(0, color='black', lw=0.8)
# Significant DMBs as tick marks
all_dmb6 = pd.concat([dmb6_p, dmb6_q]) if len(dmb6_p)+len(dmb6_q) > 0 else pd.DataFrame()
if len(all_dmb6) > 0:
    sig6 = all_dmb6[all_dmb6.get('ram_sig_99', pd.Series([False]*len(all_dmb6)))==True]
    if len(sig6):
        ax_B.scatter(sig6['mid_mb'], np.zeros(len(sig6)),
                     marker='v', color='black', s=25, zorder=5, label=f'RaM-sig DMBs (n={len(sig6)})')
add_gene_spans(ax_B, ax_B.get_ylim()[1] if ax_B.get_ylim()[1] > 0 else 0.3,
               ZOOM_GENES_IN, gene_colors)
ax_B.set_ylabel('Δβ (IMR90 − H1)', fontsize=11)
ax_B.set_title('B  Differential Methylation + RaM-significant DMBs',
               fontsize=11, fontweight='bold', loc='left')
ax_B.legend(fontsize=8, frameon=False, loc='lower right')
ax_B.spines[['top','right']].set_visible(False)
ax_B.tick_params(labelbottom=False)

# ── Panel C: CT instability score ────────────────────────────────
ax_C = fig.add_subplot(gs[2], sharex=ax_A)
if len(ct6) > 0:
    ax_C.fill_between(ct6['mid_mb'], ct6['ct_score'], alpha=0.75,
                       color='#8B0000', label='CT score (z-score)')
    ax_C.axhline(2.326, color='#FF4444', lw=2, ls='--', label='p<0.01 (z=2.33)')
    ax_C.axhline(1.645, color='orange', lw=1.5, ls=':', label='p<0.05 (z=1.64)')
add_gene_spans(ax_C, 3.0, ZOOM_GENES_IN, gene_colors)
ax_C.set_ylabel('CT Score (z)', fontsize=11)
ax_C.set_title('C  Chromothripsis Instability Score at HLA Locus',
               fontsize=11, fontweight='bold', loc='left')
ax_C.legend(fontsize=8, frameon=False, loc='upper right')
ax_C.spines[['top','right']].set_visible(False)
ax_C.tick_params(labelbottom=False)

# ── Panel D: H1 discord (Hi-C proxy structural channel) ──────────
ax_D = fig.add_subplot(gs[3], sharex=ax_A)
h1_disc   = np.abs(mat6[:,0] - mat6[:,1])
imr_disc  = np.abs(mat6[:,2] - mat6[:,3])
cross_div = np.abs(mat6[:,:2].mean(1) - mat6[:,2:4].mean(1))
ax_D.fill_between(pos6_mb, uniform_filter1d(h1_disc, ROLL),
                   alpha=0.6, color='#4393C3', label='H1 replicate discord')
ax_D.fill_between(pos6_mb, uniform_filter1d(imr_disc, ROLL),
                   alpha=0.5, color='#D6604D', label='IMR90 replicate discord')
ax_D.fill_between(pos6_mb, uniform_filter1d(cross_div, ROLL),
                   alpha=0.4, color='#6A0DAD', label='Cross-cell divergence')
add_gene_spans(ax_D, 0.5, ZOOM_GENES_IN, gene_colors)
ax_D.set_xlabel(f'{ZOOM_CHROM} Position (Mb)', fontsize=11)
ax_D.set_ylabel('Discord / Divergence', fontsize=11)
ax_D.set_title('D  V6 Hi-C Proxy Context Channels at HLA Locus\n'
               '(H1 discord | IMR90 discord | cross-cell divergence)',
               fontsize=11, fontweight='bold', loc='left')
ax_D.legend(fontsize=8, frameon=False, loc='upper right')
ax_D.spines[['top','right']].set_visible(False)

fig.suptitle(f'HLA Locus Epigenetic Analysis — chr6 {ZOOM_START/1e6:.0f}–{ZOOM_END/1e6:.0f} Mb\n'
             'Meth3D-Net V6 | H1 ESC vs IMR90 | WGBS GSE19418',
             fontsize=13, fontweight='bold', y=0.97)

fig3_pdf = f'{RESULTS_DIR}/Fig3_HLA_Zoom.pdf'
fig3_png = f'{RESULTS_DIR}/Fig3_HLA_Zoom.png'
plt.savefig(fig3_pdf, dpi=300, bbox_inches='tight')
plt.savefig(fig3_png, dpi=150, bbox_inches='tight')
plt.close()
print(f'Figure 3 saved: {fig3_pdf}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — Zip all outputs
# ═══════════════════════════════════════════════════════════════════
print('Creating V6_outputs.zip...')

ZIP_PATH = f'{RESULTS_DIR}/V6_outputs.zip'
INCLUDE_EXT = {'.json','.csv','.npy','.pt','.pdf','.png'}
EXCLUDE_EXT = set()  # include everything listed above

n_files = 0; total_mb = 0
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED,
                     compresslevel=6) as zf:
    for root, dirs, files in os.walk(RESULTS_DIR):
        # Skip the zip file itself
        dirs[:] = [d for d in dirs if d != '__pycache__']
        for fn in files:
            if fn == 'V6_outputs.zip': continue
            ext = os.path.splitext(fn)[1].lower()
            if ext not in INCLUDE_EXT: continue
            full_path = os.path.join(root, fn)
            arc_name  = os.path.relpath(full_path, RESULTS_DIR)
            sz = os.path.getsize(full_path) / 1e6
            # Skip very large .npy files (y_true/y_pred > 50 MB) to keep zip manageable
            if ext == '.npy' and sz > 50:
                print(f'  SKIP (too large): {arc_name} ({sz:.0f}MB)')
                continue
            zf.write(full_path, arc_name)
            n_files += 1; total_mb += sz

zip_size = os.path.getsize(ZIP_PATH) / 1e6
print(f'\n✅ Zip created: {ZIP_PATH}')
print(f'   Files included  : {n_files:,}')
print(f'   Raw total       : {total_mb:.0f} MB')
print(f'   Compressed size : {zip_size:.0f} MB')
print()
print('Contents by type:')
from collections import Counter
with zipfile.ZipFile(ZIP_PATH) as zf:
    ext_counts = Counter(os.path.splitext(n.filename)[1] for n in zf.infolist())
    for ext, cnt in sorted(ext_counts.items()):
        print(f'  {ext:<8} {cnt:>4} files')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 — Final status report
# ═══════════════════════════════════════════════════════════════════
print('='*60)
print('  METH3D-NET V6 — COMPLETE')
print('='*60)

if len(sum_df) > 0:
    auto = sum_df[~sum_df.chrom.isin(['chrX','chrY'])]
    print(f'  Chromosomes trained  : {len(sum_df)}/24')
    print(f'  Mean Pearson r (auto): {auto["pearson"].mean():.4f}')
    print(f'  Mean R² (auto)       : {auto["r2"].mean():.4f}')
    print(f'  Mean Hi-C coverage   : {sum_df["hic_coverage"].mean():.1f}%')
    print(f'  Total RaM-sig DMBs   : {(sum_df["dmb_p_sig"]+sum_df["dmb_q_sig"]).sum():,}')
    print(f'  Total CT-sig windows : {sum_df["ct_sig_99"].sum():,}')

print()
print('  Output files:')
for fn in ['Fig1_Genome_Accuracy.pdf','Fig2_Instability_Landscape.pdf',
           'Fig3_HLA_Zoom.pdf','V6_genome_summary.csv','V6_outputs.zip']:
    p = f'{RESULTS_DIR}/{fn}'
    if os.path.exists(p):
        print(f'    ✅ {fn:<40}  {os.path.getsize(p)/1e6:.1f} MB')
    else:
        print(f'    ❌ {fn} — NOT FOUND')

print()
print('  V6 bug fixes applied vs V5:')
print('    ✅ Hi-C proxy NaN handling (no more 0% coverage)')
print('    ✅ Top-K by |corr| not corr>0 (anticorrelated bins included)')
print('    ✅ Zero-variance bins get distance-decay fallback contacts')
print('    ✅ CT scores z-normalised (was: percentile on near-zero values)')
print('    ✅ b2i lookup uses int keys (no type mismatch)')
print('='*60)